← [Overview](00_overview.ipynb)

# Representation strategies

Clustering decides *which* periods group together; **representation** decides what each
cluster's single profile actually looks like. In tsam these are **separate steps**: the
`representation=` lever is independent of the clustering method (see
[Partitional clustering](02_clustering/01_partitional_clustering.ipynb) for that distinction). Every
method just sets a sensible default representative, which you can override freely.

This notebook covers the five strategies — `mean`, `medoid`, `maxoid`, `Distribution`,
`MinMaxMean` — and how they trade reconstruction accuracy against keeping *real* profiles or
the value *distribution*. Later steps then adjust the resulting representatives:
[Extreme periods](04_extreme_periods.ipynb) inject peaks the representative can't recover, and
[Rescaling](05_rescaling.ipynb) corrects the totals and returns the profiles to physical units.

In [ ]:
import pandas as pd
import plotly.io as pio

import tsam
from tsam import ClusterConfig, SegmentConfig
from tsam.config import Distribution, MinMaxMean

pio.renderers.default = "notebook_connected"

# Real 6-week dataset — representation differences show clearest with many periods.
raw = pd.read_csv("../../data/testdata.csv", index_col=0, parse_dates=True)
data = raw.loc["2010-01-01":"2010-02-11"]
print("real:", data.shape)

---

## The six representation strategies

| Strategy | What it picks | Use case |
|---|---|---|
| `mean` | Centroid (average) | Minimises within-cluster variance |
| `medoid` | Real period closest to centroid | Physically realistic profiles |
| `maxoid` | Real period most dissimilar to others | Spread/diversity |
| `Distribution` | Re-sorted values matching the duration curve | Preserves value distribution |
| `Distribution(preserve_minmax=True)` | Duration curve, but each column's **min & max kept exact** | Distribution *and* true peaks/troughs |
| `MinMaxMean` | Per-column mix of min/max/mean | Fine-grained control |

`Distribution` also takes a `scope` — `"cluster"` re-sorts within each cluster,
`"global"` matches the overall duration curve. Adding `preserve_minmax=True`
(the `distribution_minmax` representation) forces each column's extreme values
into the re-sorted profile, so peaks and troughs survive the re-ordering.

Default representations by method:
- `kmeans`, `averaging` → `mean`
- `kmedoids`, `hierarchical`, `contiguous` → `medoid`
- `kmaxoids` → `maxoid`

**TSAM configuration for representation strategies:**

In [ ]:
# The representation= parameter is set on ClusterConfig (or SegmentConfig).
# String shortcuts:
cfg_mean = ClusterConfig(method="hierarchical", representation="mean")
cfg_medoid = ClusterConfig(method="hierarchical", representation="medoid")
cfg_maxoid = ClusterConfig(method="kmaxoids", representation="maxoid")

# Typed objects (additional options):
cfg_dist_cluster = ClusterConfig(
    method="hierarchical",
    representation=Distribution(scope="cluster"),  # per-cluster duration curve
)
cfg_dist_global = ClusterConfig(
    method="hierarchical",
    representation=Distribution(scope="global"),  # overall duration curve
)
cfg_dist_minmax = ClusterConfig(
    method="hierarchical",
    representation=Distribution(
        scope="cluster", preserve_minmax=True
    ),  # + keep column min/max
)
cfg_minmaxmean = ClusterConfig(
    method="hierarchical",
    representation=MinMaxMean(max_columns=["Load"], min_columns=[]),
)

# SegmentConfig also accepts representation=:
cfg_seg_medoid = SegmentConfig(n_segments=6, representation="medoid")

print("mean:        ", cfg_mean)
print("medoid:      ", cfg_medoid)
print("dist_cluster:", cfg_dist_cluster)
print("dist_minmax: ", cfg_dist_minmax)
print("minmaxmean:  ", cfg_minmaxmean)
print("seg_medoid:  ", cfg_seg_medoid)

In [ ]:
# Compare representations on the real dataset
reps = {
    "mean": "mean",
    "medoid": "medoid",
    "maxoid": "maxoid",
    "distribution": Distribution(scope="cluster"),
    "distribution_global": Distribution(scope="global"),
    "distribution_minmax": Distribution(scope="cluster", preserve_minmax=True),
    "minmax_mean": MinMaxMean(max_columns=["Load"], min_columns=[]),
}
results_rep = {}
for name, rep in reps.items():
    r = tsam.aggregate(
        data,
        n_clusters=6,
        period_duration="1D",
        cluster=ClusterConfig(method="hierarchical", representation=rep),
    )
    results_rep[name] = r

summary_rep = pd.DataFrame(
    {
        name: {"weighted_rmse": round(r.accuracy.weighted_rmse, 4)}
        for name, r in results_rep.items()
    }
).T
print("Representation comparison (hierarchical k=6, real dataset):")
summary_rep

### Distribution representation — keeping the duration curve

The `mean` representative flattens the peaks of each cluster, so the reconstructed
**duration curve** sits inside the original. The `Distribution` representative instead
re-sorts values to match the cluster's duration curve, trading temporal shape for a far
better value distribution.

In [ ]:
r_mean = results_rep["mean"]
r_dist = results_rep["distribution"]

r_mean.plot.compare(
    columns=["Load"],
    mode="duration_curve",
    title="Mean representation — Load duration curve vs original",
)

In [ ]:
r_dist.plot.compare(
    columns=["Load"],
    mode="duration_curve",
    title="Distribution representation — Load duration curve vs original",
)

---

**Next:**
* [Extreme periods](04_extreme_periods.ipynb) — inject peak/trough periods into the cluster set, the next pipeline step after the representative is chosen
* [Rescaling & denormalisation](05_rescaling.ipynb) — correct the totals a non-mean representative can distort, and return to physical units
* [Representations how-to](../../how-to/representations.ipynb) — full worked examples for every strategy